# Exercise 4: Fine Tuning TerraMind for Flood Mapping

The previous two exercises used models pre trained on ordinary photographs, either not at all (Exercise 2) or through ImageNet (Exercise 3). This exercise introduces a different kind of pre trained model, a geospatial foundation model called TerraMind, which was pre trained directly on satellite data from several sensors at once, including Sentinel 1 radar imagery, Sentinel 2 optical imagery and elevation data.

We fine tune TerraMind for flood mapping using the Sen1Floods11 dataset, which provides paired Sentinel 1 and Sentinel 2 scenes together with hand labelled water masks. The exercise relies on TerraTorch, a PyTorch based library built specifically to make working with Earth observation foundation models straightforward.

## What you will learn

1. What distinguishes a geospatial foundation model such as TerraMind from a model pre trained on natural images
2. How to configure a data module that combines two different satellite modalities for the same location
3. How to assemble a segmentation model from a pre trained backbone, a set of connecting layers called necks, and a decoder
4. How to fine tune and evaluate the model using PyTorch Lightning

## Setup

TerraTorch depends on PyTorch and PyTorch Lightning. If you are running this notebook in Google Colab, select a GPU runtime before continuing, through Runtime, Change runtime type, and GPU.

In [ ]:
!pip install "terratorch>=1.2.5" -q
!pip install tensorboard "setuptools<81" -q

In [ ]:
import os
import warnings

import torch
import terratorch
import albumentations
import lightning.pytorch as pl
import matplotlib.pyplot as plt
from pathlib import Path

from terratorch.datamodules import GenericMultiModalDataModule
from terratorch.registry import BACKBONE_REGISTRY, TERRATORCH_BACKBONE_REGISTRY, TERRATORCH_DECODER_REGISTRY

warnings.filterwarnings("ignore")
pl.seed_everything(0)

## What makes TerraMind different

Ordinary pre trained vision models, including the SegFormer backbone used in Exercise 3, learn from large collections of everyday photographs. A geospatial foundation model such as TerraMind is pre trained directly on satellite data, and on more than one type of sensor at once, for example optical imagery from Sentinel 2 and radar imagery from Sentinel 1.

This matters for two reasons. First, satellite imagery differs from ordinary photographs in resolution, spectral content and geometry, so features learned from photographs transfer only partly. Second, by training across several modalities together, TerraMind learns relationships between them, for example how a given radar signature typically relates to the corresponding optical appearance. This can be useful even when only one modality is available at prediction time, and it becomes especially useful in tasks such as flood mapping, where radar is often the only usable data source because clouds block optical sensors during a flood event.

## Downloading the dataset

Sen1Floods11 pairs Sentinel 1 and Sentinel 2 scenes with hand labelled surface water masks. The version hosted on Hugging Face contains only the hand labelled subset, which keeps the download size manageable.

In [ ]:
!pip install huggingface-hub -q
if not os.path.isdir("sen1floods11_v1.1"):
    !hf download blumenstiel/Sen1Floods11 sen1floods11_v1_1.tar.gz --repo-type dataset --local-dir ./
    !tar -xzf sen1floods11_v1_1.tar.gz

dataset_path = Path("sen1floods11_v1.1")
print(sorted(os.listdir(dataset_path / "data")))

The `data` folder contains one subfolder per modality. `S2L1CHand` holds Sentinel 2 Level 1C scenes, `S1GRDHand` holds Sentinel 1 Ground Range Detected radar scenes, and `LabelHand` holds the hand labelled water masks, where a pixel is labelled 0 for no water, 1 for water, or -1 where no reliable label is available. The `splits` folder contains text files listing which scenes belong to the training, validation and test sets.

In [ ]:
print(sorted(os.listdir(dataset_path / "data" / "S2L1CHand"))[:5])
print(sorted(os.listdir(dataset_path / "splits")))

## Configuring a data module for two modalities

TerraTorch provides ready made data modules that connect a folder of files to a PyTorch Lightning training loop. Since this exercise combines two modalities, we use `GenericMultiModalDataModule`, which accepts most of its settings as dictionaries keyed by modality name, so that paths, bands and normalisation values can differ between Sentinel 1 and Sentinel 2.

We build the configuration in stages below, rather than as one large call, so that each group of settings can be explained on its own.

In [ ]:
# Which modalities to load, and which of their bands to use for quick visual inspection later.
modalities = ["S2L1C", "S1GRD"]
rgb_indices = {
    "S2L1C": [3, 2, 1],   # Red, Green, Blue among the 13 Sentinel 2 bands
    "S1GRD": [0, 1, 0],   # VV, VH, VV, since radar only has two bands
}

In [ ]:
# Where to find the images and labels for each split. All samples live in one folder,
# so separate split files are used to say which files belong to train, validation and test.
data_roots = {
    "S2L1C": dataset_path / "data/S2L1CHand",
    "S1GRD": dataset_path / "data/S1GRDHand",
}
label_root = dataset_path / "data/LabelHand"

split_files = {
    "train": dataset_path / "splits/flood_train_data.txt",
    "val": dataset_path / "splits/flood_valid_data.txt",
    "test": dataset_path / "splits/flood_test_data.txt",
}

file_patterns = {
    "S2L1C": "*_S2Hand.tif",
    "S1GRD": "*_S1Hand.tif",
}
label_pattern = "*_LabelHand.tif"

Neural networks train more reliably when their inputs are standardised, meaning centred around zero with a comparable spread across bands. The values below are the mean and standard deviation used during TerraMind's own pre training, so using them keeps our fine tuning inputs consistent with what the backbone has already learned to expect. Only Sentinel 1 GRD uses its two bands, VV and VH, as we are not selecting a subset of Sentinel 2 bands here.

In [ ]:
band_mean = {
    "S2L1C": [2357.089, 2137.385, 2018.788, 2082.986, 2295.651, 2854.537, 3122.849,
              3040.560, 3306.481, 1473.847, 506.070, 2472.825, 1838.929],
    "S1GRD": [-12.599, -20.293],
}
band_std = {
    "S2L1C": [1624.683, 1675.806, 1557.708, 1833.702, 1823.738, 1733.977, 1732.131,
              1679.732, 1727.26, 1024.687, 442.165, 1331.411, 1160.419],
    "S1GRD": [5.195, 5.890],
}

With the paths, file patterns and normalisation values defined, the data module itself can now be created. `check_stackability` is left at its default of checking that every sample has a consistent shape, and `no_label_replace` marks -1 as the value to ignore during training and evaluation, matching the convention used in the mask files.

In [ ]:
datamodule = GenericMultiModalDataModule(
    task="segmentation",
    batch_size=8,
    num_workers=2,
    num_classes=2,
    modalities=modalities,
    rgb_indices=rgb_indices,

    train_data_root=data_roots,
    train_label_data_root=label_root,
    val_data_root=data_roots,
    val_label_data_root=label_root,
    test_data_root=data_roots,
    test_label_data_root=label_root,

    train_split=split_files["train"],
    val_split=split_files["val"],
    test_split=split_files["test"],

    image_grep=file_patterns,
    label_grep=label_pattern,

    means=band_mean,
    stds=band_std,

    train_transform=[
        albumentations.D4(),  # random flips and rotations, applied identically to every modality
        albumentations.pytorch.transforms.ToTensorV2(),
    ],
    val_transform=None,
    test_transform=None,

    no_label_replace=-1,
    no_data_replace=0,
)

datamodule.setup("fit")
datamodule.setup("test")

print(f"Training samples  : {len(datamodule.train_dataset)}")
print(f"Validation samples: {len(datamodule.val_dataset)}")
print(f"Test samples       : {len(datamodule.test_dataset)}")

## Looking at a few samples

The dataset object created by TerraTorch includes a built in `plot` method that displays every modality alongside the label, with sensible scaling already applied. This is a convenient way to check that the data module has been configured correctly before spending time on training.

In [ ]:
val_dataset = datamodule.val_dataset
for index in [2, 8, 11]:
    val_dataset.plot(val_dataset[index])
plt.show()

## The TerraTorch model registry

TerraTorch keeps a registry of backbones and decoders, which makes it possible to list what is available and to build components by name. The cell below lists every TerraMind version 1 backbone, and every decoder available for pairing with them.

In [ ]:
terramind_backbones = [name for name in TERRATORCH_BACKBONE_REGISTRY if "terramind_v1" in name]
print("Available TerraMind v1 backbones:")
for name in terramind_backbones:
    print(" ", name)

print("\nAvailable decoders:")
for name in TERRATORCH_DECODER_REGISTRY:
    print(" ", name)

We use `terramind_v1_small`, a mid sized version of the backbone, configured to accept both Sentinel 2 L1C and Sentinel 1 GRD inputs. Setting `pretrained=True` downloads the pre trained weights automatically.

In [ ]:
backbone = BACKBONE_REGISTRY.build(
    "terramind_v1_small",
    modalities=modalities,
    pretrained=True,
)
print(backbone)

## Connecting the backbone to a decoder

TerraMind is a transformer based backbone, so it produces a sequence of patch tokens rather than the stack of progressively downsampled feature maps that a convolutional encoder such as the U-Net in Exercise 2 would produce. Decoders such as `UNetDecoder`, however, expect that familiar pyramidal structure. TerraTorch bridges this gap with a small chain of components called necks, placed between the backbone and the decoder.

`SelectIndices` picks out the backbone layers whose tokens will be handed to the decoder. `ReshapeTokensToImage` turns the flat sequence of tokens back into a spatial grid. `LearnedInterpolateToPyramidal` then learns how to resize these grids into the multiple resolutions a U-Net style decoder expects.

Rather than building each of these pieces manually, TerraTorch provides `SemanticSegmentationTask`, a ready made PyTorch Lightning module that assembles the backbone, the necks, the decoder and the classification head from a single configuration, and also implements the training, validation and test steps.

In [ ]:
segmentation_task = terratorch.tasks.SemanticSegmentationTask(
    model_factory="EncoderDecoderFactory",
    model_args={
        "backbone": "terramind_v1_small",
        "backbone_pretrained": True,
        "backbone_modalities": modalities,

        "necks": [
            {"name": "SelectIndices", "indices": [2, 5, 8, 11]},  # matches terramind_v1_small
            {"name": "ReshapeTokensToImage", "remove_cls_token": False},
            {"name": "LearnedInterpolateToPyramidal"},
        ],

        "decoder": "UNetDecoder",
        "decoder_channels": [256, 128, 64, 32],

        "head_dropout": 0.1,
        "num_classes": 2,
    },

    loss="dice",
    optimizer="AdamW",
    lr=2e-5,
    scheduler="ReduceLROnPlateau",
    scheduler_hparams={"factor": 0.5, "patience": 5},

    ignore_index=-1,
    freeze_backbone=True,   # kept frozen here to make this demonstration run quickly
    freeze_decoder=False,
    plot_on_val=True,
    class_names=["background", "water"],
    class_weights=[0.3, 0.7],
)

Two settings above are worth highlighting. The loss function is set to Dice loss, which is generally a good default for binary segmentation tasks with a class imbalance, since it directly rewards overlap between predicted and true water pixels rather than treating every pixel equally as cross entropy would. `freeze_backbone` is set to `True` so that only the decoder is trained in this demonstration, which keeps the run short. Fine tuning the backbone as well typically gives better results, at the cost of a longer training time, and is recommended once you move beyond this introductory exercise.

## Training

PyTorch Lightning's `Trainer` handles the training loop, moving data and the model to the GPU, running the optimiser, and calling into the checkpoint callback. The checkpoint callback below saves the version of the model that reaches the highest validation mean Intersection over Union, rather than simply the version from the final epoch.

In [ ]:
checkpoint_directory = "output/terramind_small_sen1floods11/checkpoints/"

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath=checkpoint_directory,
    monitor="val/mIoU",
    mode="max",
    filename="best-mIoU",
    save_weights_only=True,
)

trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    max_epochs=3,
    log_every_n_steps=1,
    callbacks=[checkpoint_callback],
    default_root_dir="output/terramind_small_sen1floods11/",
)

In [ ]:
trainer.fit(segmentation_task, datamodule=datamodule)

## Testing the fine tuned model

With training complete, we load the checkpoint that reached the best validation score and evaluate it on the held out test split. Three epochs are enough to demonstrate the pipeline end to end, but not enough to reach the model's best possible accuracy. Longer training, and unfreezing the backbone, will noticeably improve these numbers.

In [ ]:
best_checkpoint_path = os.path.join(checkpoint_directory, "best-mIoU.ckpt")
trainer.test(segmentation_task, datamodule=datamodule, ckpt_path=best_checkpoint_path)

## Visualising predictions

To inspect the fine tuned model directly, we load its weights, run it on a batch from the test set, and use the dataset's `plot` method to display each modality, the ground truth mask and the prediction together.

In [ ]:
trained_task = terratorch.tasks.SemanticSegmentationTask.load_from_checkpoint(
    best_checkpoint_path,
    model_factory=segmentation_task.hparams.model_factory,
    model_args=segmentation_task.hparams.model_args,
)
trained_task.eval()

test_loader = datamodule.test_dataloader()
test_dataset = datamodule.test_dataset

with torch.no_grad():
    batch = next(iter(test_loader))
    original_images = {modality: value.clone() for modality, value in batch["image"].items()}

    batch = datamodule.aug(batch)
    inputs = {modality: value.to(trained_task.device) for modality, value in batch["image"].items()}

    outputs = trained_task(inputs)
    predictions = torch.argmax(outputs.output, dim=1).cpu().numpy()

for sample_index in range(5):
    sample = {
        "S2L1C": original_images["S2L1C"][sample_index].cpu(),
        "S1GRD": original_images["S1GRD"][sample_index].cpu(),
        "mask": batch["mask"][sample_index],
        "prediction": predictions[sample_index],
    }
    test_dataset.plot(sample)
    plt.show()

## Summary

This exercise fine tuned a geospatial foundation model that was pre trained on multiple satellite modalities at once, rather than a model pre trained on ordinary photographs. The overall recipe followed the same shape as Exercise 3, loading pre trained weights and fine tuning them on labelled data, but the surrounding pipeline needed extra components, namely the multimodal data module and the necks that reshape transformer tokens into the pyramidal structure a segmentation decoder expects.

Across Exercises 2 to 4, three distinct strategies have now been demonstrated for building a segmentation model: training a convolutional network entirely from scratch, fine tuning a transformer pre trained on natural images, and fine tuning a transformer pre trained directly on satellite data. Exercise 5 introduces the metrics and diagnostic tools needed to compare approaches such as these in a rigorous way.